# Hypothesis: The proportion of car crashes occurring between 4 PM and 8 PM is greater than 50%.

# Problem statement: I want to determine whether more than 50% of car crashes occur between 4 PM and 8 PM.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.remote("sc://localhost").getOrCreate()
spark



In [2]:
import os

os.getcwd()

'/home/analytics4220/Anita'

In [3]:
df = spark.read.csv("/home/analytics4220/Anita/Time_of_driving_data.csv", header=True) 
df

DataFrame[Crash Date/Time: string, Route Type: string, Collision Type: string]

In [19]:
df.printSchema()

root
 |-- Crash Date/Time: string (nullable = true)
 |-- Route Type: string (nullable = true)
 |-- Collision Type: string (nullable = true)
 |-- Crash_Timestamp: timestamp (nullable = true)
 |-- hour: integer (nullable = true)
 |-- is_peak: integer (nullable = false)



In [4]:
df.select("Crash Date/Time").show(10)

+--------------------+
|     Crash Date/Time|
+--------------------+
|08/21/2025 05:21:...|
|08/22/2025 10:44:...|
|07/25/2025 11:55:...|
|08/22/2025 10:36:...|
|08/03/2025 02:10:...|
|08/19/2025 09:50:...|
|08/23/2025 11:50:...|
|08/21/2025 10:45:...|
|08/20/2025 03:15:...|
|08/04/2025 03:47:...|
+--------------------+
only showing top 10 rows


In [12]:
from pyspark.sql.functions import expr

df = df.withColumn(
    "Crash_Timestamp",
    expr("try_to_timestamp(`Crash Date/Time`, 'MM/dd/yyyy hh:mm:ss a')")
)

In [13]:
df = df.filter(df.Crash_Timestamp.isNotNull())

In [14]:
from pyspark.sql.functions import hour

df = df.withColumn("hour", hour("Crash_Timestamp"))

In [15]:
from pyspark.sql.functions import when

df = df.withColumn(
    "is_peak",
    when((df.hour >= 16) & (df.hour < 20), 1).otherwise(0)
)

In [9]:
df.select("Crash Date/Time", "hour", "is_peak").show(10)

+--------------------+----+-------+
|     Crash Date/Time|hour|is_peak|
+--------------------+----+-------+
|08/21/2025 05:21:...|  17|      1|
|08/22/2025 10:44:...|  10|      0|
|07/25/2025 11:55:...|  11|      0|
|08/22/2025 10:36:...|  10|      0|
|08/03/2025 02:10:...|  14|      0|
|08/19/2025 09:50:...|   9|      0|
|08/23/2025 11:50:...|  11|      0|
|08/21/2025 10:45:...|  22|      0|
|08/20/2025 03:15:...|  15|      0|
|08/04/2025 03:47:...|  15|      0|
+--------------------+----+-------+
only showing top 10 rows


In [16]:
from pyspark.sql.functions import avg

df.select(avg("is_peak").alias("proportion_peak")).show()

+-------------------+
|    proportion_peak|
+-------------------+
|0.27748235338644667|
+-------------------+



#### Final Conclusion: The goal of this analysis was to determine whether more than 50% of car crashes occur between 4 PM and 8 PM. Using Spark, the proportion of crashes occurring in this time range was calculated to be approximately 27.7%. Since this value is less than 0.5, we fail to reject the null hypothesis. Therefore, there is insufficient evidence to conclude that most crashes occur between 4 PM and 8 PM in this dataset.

#### The conclusion differs from the previous assignment in that the hypothesis was not supported in either case. However, this does not indicate bias, but rather that the observed data does not support the initial assumptions. Each dataset and hypothesis addresses a different aspect of traffic patterns, so the results are not directly comparable.